<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/01_operaciones_vectores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 01 &middot; Operaciones básicas con vectores y matrices

**Módulo 1 — Álgebra lineal y geometría diferencial**

Un modelo de inteligencia artificial no "ve" imágenes ni "lee" texto: recibe **vectores** y los
transforma con **matrices**. Una foto es un vector de píxeles, una palabra es un vector de
cientos de números, y cada capa de una red neuronal es, en el fondo, una multiplicación de
matrices. Todo lo que hagas después en IA se apoya en las operaciones de este notebook.

## Al terminar vas a poder

- Calcular e interpretar el **producto punto**, y saber por qué es el corazón de una neurona.
- Medir el **tamaño** de un vector (la norma) y **normalizarlo**.
- Medir el **parecido** entre dos vectores con la similitud coseno, la misma que usan los buscadores.
- Leer una **matriz como una transformación** del espacio y saber qué le hace al área.
- Encontrar **valores y vectores propios**, y entender qué dirección revelan en un conjunto de datos.

## Qué necesitas saber antes

Álgebra de bachillerato (coordenadas, raíz cuadrada, algo de trigonometría) y saber leer código
sencillo de Python. Nada más: lo demás se construye aquí.

## Cómo usar este notebook

1. Si entraste con el botón **Open in Colab** de arriba, no tienes que instalar nada.
2. Ejecuta las celdas en orden con `Shift + Enter`.
3. **Cambia los números y vuelve a ejecutar.** Ahí es donde de verdad se aprende.
4. Al final hay una sección **Tu turno** con ejercicios para resolver.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Menos decimales y sin notacion cientifica: los resultados se leen mejor.
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5, 5)

print("numpy:", np.__version__)

---

## 1. Un vector es una lista de números con dirección

En el papel escribimos $\vec{u} = (3, 4)$; en NumPy es un arreglo. Con dos componentes es una
flecha en el plano y la podemos dibujar. Con tres, una flecha en el espacio. Los vectores con
los que trabaja un modelo de lenguaje tienen cientos o miles de componentes y ya no hay forma
de dibujarlos, pero **las operaciones son exactamente las mismas** que vas a ver aquí.

In [ ]:
u = np.array([3.0, 4.0])
v = np.array([1.0, 0.0])

print("u =", u)
print("v =", v)
print("dimension de u:", u.shape)   # (2,) = dos componentes
print("u + v =", u + v)
print("2 * u =", 2 * u)

In [ ]:
def dibujar_vectores(vectores, etiquetas, limite=6, titulo=""):
    """Dibuja vectores como flechas que salen del origen."""
    colores = plt.cm.tab10.colors
    fig, ax = plt.subplots()
    for i, (vec, nombre) in enumerate(zip(vectores, etiquetas)):
        ax.quiver(0, 0, vec[0], vec[1], angles="xy", scale_units="xy", scale=1,
                  color=colores[i % len(colores)], width=0.012)
        ax.text(vec[0] * 1.06, vec[1] * 1.06, nombre, fontsize=12,
                color=colores[i % len(colores)])
    ax.set_xlim(-limite, limite)
    ax.set_ylim(-limite, limite)
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_aspect("equal")
    ax.set_title(titulo)
    plt.show()


dibujar_vectores([u, v], ["u", "v"], titulo="Los vectores u y v")

---

## 2. Producto punto: la operación más importante de todas

El producto punto multiplica componente con componente y suma todo:

$$\vec{u} \cdot \vec{v} = u_1 v_1 + u_2 v_2 + \dots + u_n v_n$$

Fíjate en algo: entran dos vectores y sale **un solo número**, no un vector. Ese número dice
qué tanto apuntan los dos hacia el mismo lado. Es positivo si van en la misma dirección, cero
si son perpendiculares y negativo si se oponen.

In [ ]:
# A mano, para ver que no hay magia:
producto_manual = sum(u[i] * v[i] for i in range(len(u)))

# Y con NumPy, que es lo que usaras siempre:
producto_numpy = np.dot(u, v)      # equivalente: u @ v

print("a mano :", producto_manual)
print("numpy  :", producto_numpy)
print("iguales:", np.isclose(producto_manual, producto_numpy))

### Por qué importa en IA

Una **neurona artificial** no hace nada más que esto: toma las entradas $\vec{x}$, las pondera
con los pesos $\vec{w}$, suma un sesgo $b$ y pasa el resultado por una función de activación.

$$\text{salida} = f(\vec{w} \cdot \vec{x} + b)$$

Ese punto de en medio es el producto punto que acabas de calcular. Una red neuronal con
millones de parámetros son millones de productos punto, organizados en matrices para que la
computadora los haga todos de golpe.

In [ ]:
def neurona(x, w, b):
    """Una neurona con activacion ReLU: el ladrillo basico de una red."""
    z = np.dot(w, x) + b
    return max(0.0, z)             # ReLU: deja pasar lo positivo, corta lo negativo


x = np.array([0.5, 2.0, -1.0])     # entradas
w = np.array([1.5, -0.5, 2.0])     # pesos, que en un modelo real se aprenden entrenando
b = 0.3                            # sesgo

print("z crudo      :", np.dot(w, x) + b)
print("salida (ReLU):", neurona(x, w, b))

---

## 3. Norma: el tamaño de un vector

La norma es la longitud de la flecha y sale del teorema de Pitágoras de toda la vida:

$$\|\vec{u}\| = \sqrt{u_1^2 + u_2^2 + \dots + u_n^2} = \sqrt{\vec{u} \cdot \vec{u}}$$

**Normalizar** es dividir el vector entre su norma para dejarlo de longitud 1 conservando su
dirección. En IA se hace constantemente: así se comparan vectores por su *orientación*, sin
que el tamaño meta ruido.

In [ ]:
norma_manual = np.sqrt(np.dot(u, u))
norma_numpy = np.linalg.norm(u)

print("norma de u (a mano):", norma_manual)
print("norma de u (numpy) :", norma_numpy)

u_unitario = u / np.linalg.norm(u)
print("u normalizado      :", u_unitario)
print("su norma           :", np.linalg.norm(u_unitario), "<- siempre 1")

---

## 4. Ángulo y similitud coseno

Juntando el producto punto y la norma se despeja el ángulo entre dos vectores:

$$\cos\theta = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\|\,\|\vec{v}\|}$$

Ese coseno tiene nombre propio en IA: **similitud coseno**. Vale 1 si los vectores apuntan
igual, 0 si son perpendiculares y -1 si son opuestos. Es la medida con la que un buscador
semántico decide qué documento se parece más a tu pregunta, y con la que un sistema de
recomendación encuentra usuarios de gustos parecidos.

In [ ]:
def similitud_coseno(a, b):
    """Coseno del angulo entre dos vectores: de -1 (opuestos) a 1 (identicos)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def angulo_entre(a, b, en_grados=True):
    """Angulo entre dos vectores.

    El clip corrige errores de redondeo: sin el, un coseno de 1.0000000002
    haria fallar a arccos.
    """
    cos = np.clip(similitud_coseno(a, b), -1.0, 1.0)
    angulo = np.arccos(cos)
    return np.degrees(angulo) if en_grados else angulo


print("similitud u,v :", similitud_coseno(u, v))
print("angulo u,v    :", angulo_entre(u, v), "grados")

In [ ]:
# Embeddings de juguete: cada palabra es un vector.
# En un modelo real tendrian cientos de dimensiones y saldrian del entrenamiento.
embeddings = {
    "gato": np.array([0.9, 0.8, 0.1]),
    "perro": np.array([0.8, 0.9, 0.2]),
    "automovil": np.array([0.1, 0.2, 0.95]),
}

consulta = "gato"
print(f"Que se parece mas a '{consulta}'?\n")
for palabra, vec in embeddings.items():
    if palabra == consulta:
        continue
    s = similitud_coseno(embeddings[consulta], vec)
    ang = angulo_entre(embeddings[consulta], vec)
    print(f"  {palabra:10} similitud = {s:.4f}   angulo = {ang:5.1f} grados")

---

## 5. Una matriz es una transformación del espacio

Multiplicar una matriz por un vector, $A\vec{u}$, produce otro vector: la matriz **mueve** al
vector. Estirar, comprimir, rotar y sesgar son todas multiplicaciones por una matriz.

Una capa de una red neuronal hace exactamente esto con los datos: los lleva a un espacio nuevo
donde el problema resulta más fácil de resolver.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])

print("A =\n", A)
print("\nA @ u =", A @ u)          # @ es el producto matricial
print("A @ v =", A @ v)

dibujar_vectores([u, A @ u], ["u", "A@u"], limite=16,
                 titulo="La matriz A transforma a u")

In [ ]:
# Y como deforma A a TODO el plano: le seguimos la pista al cuadrado unitario.
cuadrado = np.array([[0, 1, 1, 0, 0],
                     [0, 0, 1, 1, 0]])         # 4 esquinas, repitiendo la primera al cerrar
transformado = A @ cuadrado

fig, ax = plt.subplots()
ax.plot(cuadrado[0], cuadrado[1], "o-", label="cuadrado unitario (area 1)")
ax.fill(cuadrado[0], cuadrado[1], alpha=0.2)
ax.plot(transformado[0], transformado[1], "o-", label="despues de aplicar A")
ax.fill(transformado[0], transformado[1], alpha=0.2)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.3)
ax.set_aspect("equal")
ax.legend()
ax.set_title("A deforma el plano")
plt.show()

---

## 6. Determinante: cuánto cambia el área

El determinante de $A$ es el factor por el que se multiplica el área al aplicar la
transformación. En la figura de arriba, el cuadrado de área 1 se convirtió en un paralelogramo
cuya área es justamente $|\det A|$.

Si el determinante es **0**, la transformación aplasta el plano sobre una línea: se pierde
información y la matriz **no tiene inversa**. No es un caso raro de libro de texto: pasa
siempre que dos variables de tus datos son redundantes.

In [ ]:
# Nota: la computadora responde 5.000000000000001 en vez de 5. No es un error tuyo,
# es como se guardan los decimales en binario. Por eso redondeamos al imprimir y
# comparamos con np.isclose en vez de con ==. El tema completo esta en el modulo 4.
det_A = np.linalg.det(A)

print(f"det(A) = {det_A:.4f}")
print(f"-> el area se multiplica por {abs(det_A):.4f}")

singular = np.array([[1.0, 2.0],
                     [2.0, 4.0]])              # la segunda fila es el doble de la primera
print(f"\ndet(singular) = {np.linalg.det(singular):.4f}",
      "-> aplasta el plano sobre una linea, no es invertible")

A_inv = np.linalg.inv(A)
print("\nA_inv @ A =\n", A_inv @ A, "\n<- la identidad, como debe ser")

---

## 7. Valores y vectores propios

Casi cualquier vector cambia de dirección al multiplicarlo por $A$. Unos pocos, los **vectores
propios**, sólo se estiran o se encogen, sin girar:

$$A\vec{p} = \lambda\vec{p}$$

El factor $\lambda$ es el **valor propio** que le corresponde. Son los ejes naturales de la
transformación y contestan la pregunta "¿en qué direcciones actúa realmente esta matriz?".

En IA aparecen en **PCA** (análisis de componentes principales): los vectores propios de la
matriz de covarianza de tus datos apuntan hacia donde los datos más varían, así que quedarte
sólo con los primeros te deja reducir dimensiones perdiendo lo mínimo.

In [ ]:
valores, vectores = np.linalg.eig(A)   # cada COLUMNA de `vectores` es un vector propio

print("valores propios  :", valores)
print("vectores propios :\n", vectores)

# Verificamos la definicion: A @ p tiene que dar lo mismo que lambda * p
for i in range(len(valores)):
    p, lam = vectores[:, i], valores[i]
    print(f"\nvector propio {i + 1}: {p}   (lambda = {lam:.4f})")
    print("  A @ p     =", A @ p)
    print("  lambda*p  =", lam * p)
    print("  coinciden :", np.allclose(A @ p, lam * p))

In [ ]:
# Un vector propio no gira; cualquier otro si.
p1 = vectores[:, 0]
otro = np.array([1.0, 0.0])

dibujar_vectores(
    [p1, A @ p1, otro, A @ otro],
    ["p1", "A@p1", "otro", "A@otro"],
    limite=5,
    titulo="p1 solo se escala; 'otro' ademas gira",
)

---

## Tu turno

Resuelve en la celda de abajo. A propósito no hay respuestas al final: la idea es que
**verifiques cada resultado con código**, comparando tu versión contra la de NumPy.

1. Define $\vec{a} = (1, 2, 2)$ y $\vec{b} = (2, 0, -1)$. Calcula su producto punto, sus normas
   y el ángulo entre ellos. ¿Qué te dice el signo del producto punto que obtuviste?
2. Escribe tu propia función `mi_norma(v)` **sin usar** `np.linalg.norm` y comprueba con
   `np.isclose` que coincide con la de NumPy para varios vectores.
3. Encuentra a mano un vector perpendicular a $(3, 4)$ y confirma con código que su producto
   punto da 0. ¿Cuántos vectores perpendiculares a $(3,4)$ existen?
4. Construye la matriz de rotación de 90 grados, $R = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$.
   Aplícala a `u` y dibuja el resultado con `dibujar_vectores`. ¿Cuánto vale $\det R$ y por qué
   tiene sentido ese valor?
5. Calcula los valores propios de `R`. ¿Por qué salen números complejos? Piénsalo con la
   definición en la mano: ¿hay alguna dirección real que una rotación de 90 grados no gire?

In [ ]:
# Tu codigo aqui.
a = np.array([1.0, 2.0, 2.0])
b = np.array([2.0, 0.0, -1.0])

# 1.


---

## Resumen

| Operación | En NumPy | Para qué sirve en IA |
|---|---|---|
| Producto punto | `np.dot(u, v)` o `u @ v` | Es lo que calcula cada neurona |
| Norma | `np.linalg.norm(u)` | Medir tamaño, normalizar, regularizar |
| Similitud coseno | `u @ v / (norma_u * norma_v)` | Comparar embeddings, búsqueda semántica |
| Producto matricial | `A @ u` | Una capa de una red transforma los datos |
| Determinante | `np.linalg.det(A)` | Detectar información redundante |
| Valores propios | `np.linalg.eig(A)` | PCA y reducción de dimensiones |

## Qué sigue

- El mismo tema en MATLAB/Octave: [`ej01_operaciones_vectores.m`](../../matlab/01-algebra-lineal/ej01_operaciones_vectores.m)
- La versión en script, más corta y sin explicaciones: [`01_operaciones_vectores.py`](01_operaciones_vectores.py)
- Los demás módulos, en el [README del repositorio](../../README.md)

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*